# Model Definition and Evaluation
## Table of Contents
1. [Model Selection](#model-selection)
2. [Feature Engineering](#feature-engineering)
3. [Hyperparameter Tuning](#hyperparameter-tuning)
4. [Implementation](#implementation)
5. [Evaluation Metrics](#evaluation-metrics)
6. [Comparative Analysis](#comparative-analysis)


In [36]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
# Import models you're considering


## Model Selection

[Discuss the type(s) of models you consider for this task, and justify the selection.]



## Feature Engineering

[Describe any additional feature engineering you've performed beyond what was done for the baseline model.]


In [37]:
# Load the dataset
# Replace 'your_dataset.csv' with the path to your actual dataset
df = pd.read_csv("../1_DatasetCharacteristics/hanabi_dataset_clean.csv")

# Perform any feature engineering steps
# Example: df['new_feature'] = df['feature1'] + df['feature2']
# Feature and target variable selection
X = df.iloc[:, 2:]
y = df['final_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42
)

## Hyperparameter Tuning

[Discuss any hyperparameter tuning methods you've applied, such as Grid Search or Random Search, and the rationale behind them.]


In [38]:
# Implement hyperparameter tuning
# Example using GridSearchCV with a DecisionTreeClassifier
# param_grid = {'max_depth': [2, 4, 6, 8]}
# grid_search = GridSearchCV(DecisionTreeClassifier(), param_grid, cv=5)
# grid_search.fit(X_train, y_train)


## Implementation

[Implement the final model(s) you've selected based on the above steps.]


In [39]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)

#only transform the validation and test sets, no fitting -> avoiding data leakage
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

In [40]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [41]:
num_epochs = 100

In [42]:
class HanabiScorePredictor(nn.Module):
    def __init__(self, input_dim):
        super(HanabiScorePredictor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.network(x)

model = HanabiScorePredictor(input_dim= X_train.shape[1])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [43]:
num_epochs = 100

train_losses = []
val_losses = []

for epoch in range(num_epochs):

    # Training
    model.train()
    running_train_loss = 0.0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    epoch_train_loss = running_train_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:

            outputs = model(X_batch)
            val_loss = criterion(outputs, y_batch)

            running_val_loss += val_loss.item()

    epoch_val_loss = running_val_loss / len(val_loader)
    val_losses.append(epoch_val_loss)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Val Loss: {epoch_val_loss:.4f}"
    )

Epoch [1/100] Train Loss: 166.9662 Val Loss: 113.9301
Epoch [2/100] Train Loss: 54.8610 Val Loss: 37.2311
Epoch [3/100] Train Loss: 36.8811 Val Loss: 34.7719
Epoch [4/100] Train Loss: 35.8476 Val Loss: 34.9994
Epoch [5/100] Train Loss: 35.2908 Val Loss: 33.7284
Epoch [6/100] Train Loss: 33.7541 Val Loss: 33.8665
Epoch [7/100] Train Loss: 34.2649 Val Loss: 34.5621
Epoch [8/100] Train Loss: 34.4258 Val Loss: 33.4293
Epoch [9/100] Train Loss: 33.4436 Val Loss: 33.5000
Epoch [10/100] Train Loss: 33.7132 Val Loss: 33.5945
Epoch [11/100] Train Loss: 33.5909 Val Loss: 33.4338
Epoch [12/100] Train Loss: 33.1150 Val Loss: 33.5352
Epoch [13/100] Train Loss: 33.2561 Val Loss: 33.5205
Epoch [14/100] Train Loss: 32.8549 Val Loss: 33.8599
Epoch [15/100] Train Loss: 32.3444 Val Loss: 33.2492
Epoch [16/100] Train Loss: 32.3748 Val Loss: 33.7952
Epoch [17/100] Train Loss: 31.8369 Val Loss: 33.7327
Epoch [18/100] Train Loss: 32.8475 Val Loss: 33.4062
Epoch [19/100] Train Loss: 30.6110 Val Loss: 33.8459


In [44]:
model.eval()

with torch.no_grad():
    predictions = model(X_test_tensor)

    mse = nn.MSELoss()(predictions, y_test_tensor)
    rmse = torch.sqrt(mse)

print(f"Test MSE: {mse.item():.4f}")
print(f"Test RMSE: {rmse.item():.4f}")

Test MSE: 40.1854
Test RMSE: 6.3392


## Evaluation Metrics

[Clearly specify which metrics you'll use to evaluate the model performance, and why you've chosen these metrics.]


In [45]:
# Evaluate the model using your chosen metrics
# Example for classification
# y_pred = model.predict(X_test)
# print(classification_report(y_test, y_pred))

# Example for regression
# mse = mean_squared_error(y_test, y_pred)

# Your evaluation code here


## Comparative Analysis

[Compare the performance of your model(s) against the baseline model. Discuss any improvements or setbacks and the reasons behind them.]


In [46]:
# Comparative Analysis code (if applicable)
# Example: comparing accuracy of the baseline model and the new model
# print(f"Baseline Model Accuracy: {baseline_accuracy}, New Model Accuracy: {new_model_accuracy}")
